# Netzparameter und Busspannungen

## Importe

In [1]:
import base_net as bn
import pandapower as pp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In die Datei `base_net.py` wurden folgende Funktionen ausgelagert:

- `build_base_net()`: erzeugt das Ausgangsnetz (Baseline) und gibt das Netz sowie die Indizes aller Elemente (`idx`) zurück. So startet jeder Test von einem frischen, identischen Netz.
- `build_110kV_net()`, `build_110kV_net_simple()`, `build_2_load_net()`: erzeugen die Netzvarianten für Aufgabe 6.
- `vary_parameter(bauteil, parameter, werte, ausgaben)`: ändert einen Parameter eines Bauteils nacheinander auf alle angegebenen Werte, rechnet für jeden Wert einen Lastfluss auf einer frischen Netzkopie, prüft die Konvergenz und gibt die gewünschten Ergebnisse als Tabelle zurück. Der Ausgangswert (Baseline) wird automatisch mitgerechnet und markiert.

## Netzwerk
Externes Netz → Bus 1 (20 kV) → Transformator → Bus 2 (0,4 kV) → Kabel → Bus 3 (0,4 kV) → Last

| Komponente | pandapower-Element | Wichtige Daten | Funktion |
|---|---|---|---|
| Externes Netz | `ext_grid` an Bus 1 | `vm_pu` = 1,02, Winkel 0° | Vorgelagertes Mittelspannungsnetz als starre Spannungsquelle. Es ist der Slack-Bus: Er gibt Spannung und Winkelreferenz vor und gleicht die Leistungsbilanz (Last + Verluste) aus. |
| Bus 1 | `bus`, 20 kV | Nennspannung 20 kV | Mittelspannungs-Sammelschiene, Oberspannungsseite des Trafos |
| Transformator | `trafo`, Typ „0.4 MVA 20/0.4 kV“ | S_r = 0,4 MVA, v_k = 6 %, v_kr = 1,425 %, Schaltgruppe Dyn5, Stufensteller ±2 × 2,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 2 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Niederspannungs-Sammelschiene der Ortsnetzstation |
| Kabel | `line`, Typ „NAYY 4x50 SE“ | 0,1 km, r = 0,642 Ω/km, x = 0,083 Ω/km, c = 210 nF/km, I_max = 142 A, 1 System | Niederspannungs-Erdkabel (Aluminium, 50 mm²) zum Verbraucher |
| Bus 3 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Netzknoten am Ende des Kabels, Anschlusspunkt des Verbrauchers |
| Last | `load` an Bus 3 | P = 0,05 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |

Die Wirkleistung der Last wurde gegenüber dem Tutorial von 0,1 MW auf 0,05 MW reduziert. Mit 0,1 MW fließen etwa 148 A über das Kabel NAYY 4x50 SE, dessen thermisch zulässiger Dauerstrom 142 A beträgt (Auslastung 104,5 %). Ein Dauerbetrieb oberhalb von 100 % ist nicht zulässig. Mit 0,05 MW liegt die Auslastung bei 51 %, sodass in der Baseline ein realistischer, zulässiger Betriebspunkt vorliegt und Parametervariationen in beide Richtungen möglich sind.

## pu-Größen

Alle Spannungen werden in der Einheit `pu` (per unit) dargestellt. Diese ist auf die Nennspannung des jeweiligen Busses (`vn_kv`) bezogen: 1,0 pu entspricht an Bus 1 20 kV und an Bus 2/3 400 V.
### Vorteile
Die Vorteile daraus sind vor allem, dass man direkt sieht, ob an einem Bus die Spannung zu hoch oder zu niedrig ist, und entsprechend regeln kann, ohne direkt zu wissen, wie hoch die Spannung an dem entsprechenden Bus eigentlich sein sollte.

Außerdem:
- Spannungen auf verschiedenen Spannungsebenen sind direkt vergleichbar, ohne über die Übersetzung umzurechnen.
- Grenzwerte lassen sich einheitlich formulieren, z. B. das Spannungsband 0,9–1,1 pu nach EN 50160.
- Bei Nennübersetzung fällt der ideale Transformator aus dem Ersatzschaltbild heraus. Impedanzen von Trafo und Leitung lassen sich direkt vergleichen.

## Untersuchte Busse

| Bus | Spannung | Funktion |
|---|---|---|
| Bus 1 | 20 kV | Slack-Bus am externen Netz, Spannung fest 1,02 pu, Winkelreferenz 0° |
| Bus 2 | 0,4 kV | NS-Sammelschiene der Ortsnetzstation. Hier wirkt sich der Transformator aus. |
| Bus 3 | 0,4 kV | Ende des Kabels, Anschlusspunkt der Last. Hier wirken Transformator und Leitung zusammen, die Spannung ist am niedrigsten. |

Die Winkel an Bus 2 und Bus 3 liegen bei etwa −150°. Das ist kein Spannungsfall, sondern die Phasendrehung der Schaltgruppe **Dyn5** (5 × 30° = 150°, Spalte `shift_degree`). In den Auswertungen wird deshalb der Winkel ohne diesen Offset (`va_degree + 150`) bzw. die Differenz Bus 3 − Bus 2 betrachtet.

## Konvergenz

`pp.runpp()` löst die nichtlinearen Leistungsflussgleichungen iterativ mit dem **Newton-Raphson-Verfahren**. Ausgehend von einem Startwert werden Spannungsbeträge und Winkel so lange korrigiert, bis die Abweichung zwischen berechneter und vorgegebener Leistung an jedem Bus unter einer Toleranz liegt.

**Konvergenz** bedeutet, dass diese Bedingung innerhalb der maximalen Iterationszahl erreicht wurde und eine gültige Lösung vorliegt. Konvergiert der Lastfluss nicht, existiert häufig keine physikalische Lösung, z. B. weil die angeforderte Leistung größer ist als die maximal übertragbare Leistung. pandapower wirft dann `LoadflowNotConverged`. `vary_parameter()` fängt diesen Fall ab, markiert ihn in der Spalte `konvergiert` und lässt die Ergebniswerte leer, sodass nicht konvergierte Fälle nicht als gültige Spannungswerte erscheinen.

Wichtig: Ein **konvergierter** Lastfluss ist noch kein **zulässiger** Betriebspunkt. Spannungsband und Auslastung müssen zusätzlich geprüft werden.

## Datenstruktur

In [2]:
net, idx = bn.build_base_net()

In [3]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,20.0,b,None,True,None
1,Bus 2,0.4,b,None,True,None
2,Bus 3,0.4,b,None,True,None


In [4]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,...,tap_step_degree,tap_pos,tap_changer_type,id_characteristic_table,tap_dependency_table,parallel,df,in_service,vector_group,oltc
0,Trafo,0.4 MVA 20/0.4 kV,0,1,0.4,20.0,0.4,6.0,1.425,1.35,...,0.0,0.0,Ratio,<NA>,False,1,1.0,True,Dyn5,False


In [5]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line,NAYY 4x50 SE,1,2,0.1,0.642,0.083,210.0,0.0,0.142,1.0,1,cs,True,None


In [6]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type
0,Load,2,0.05,0.005,0.0,0.0,0.0,0.0,NaN,1.0,True,wye


## Ausgangslage

In [7]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052513,-0.005511
1,1.017429,-150.405971,0.000000,0.000000
2,0.997047,-150.439166,0.050000,0.005000


In [8]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051019,0.005131,-0.05,-0.005,0.001019,0.000131,0.072743,0.072744,0.072744,1.017429,-150.405971,0.997047,-150.439166,51.227854


In [9]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.052513,0.005511,-0.051019,-0.005131,0.001494,0.00038,0.001494,0.072743,1.02,0.0,1.017429,-150.405971,12.941537


In [10]:
net.res_load

,p_mw,q_mvar
0,0.05,0.005


### Auswertung Baseline

| Größe | Wert |
|---|---|
| vm Bus 2 / Bus 3 | 1,0174 pu / 0,9970 pu |
| Spannungsfall Bus 2 → 3 | 0,0204 pu |
| Strom Leitung | 72,7 A |
| Auslastung Leitung / Trafo | 51,2 % / 12,9 % |
| Verluste Leitung / Trafo | 1,02 kW / 1,49 kW |

Alle Werte liegen im zulässigen Bereich. Die Trafoverluste sind größer als die Leitungsverluste, weil bei geringer Auslastung die lastunabhängigen **Eisenverluste** (`pfe_kw` = 1,35 kW) dominieren.

### Grundlage für alle Auswertungen

Für den Spannungsfall über einer Impedanz R + jX gilt näherungsweise:

- Längsspannungsfall (bestimmt den **Betrag**): ΔU ≈ (R·P + X·Q) / U
- Querspannungsfall (bestimmt den **Winkel**): δU ≈ (X·P − R·Q) / U

Der **Transformator ist X-dominiert**: v_kx = √(6² − 1,425²) ≈ 5,83 % gegenüber v_kr = 1,425 %, also X/R ≈ 4.
Das **NAYY-Kabel ist R-dominiert**: R/X = 0,642 / 0,083 ≈ 7,7.

## Ausgabegrößen für alle Tests
Jede Zeile ist `[Quelle, Spaltenname]`. Die Quelle hat das Format `ergebnistabelle.spalte.bauteil`.

In [11]:
OUTPUTS = [
    ["res_bus.vm_pu.bus2",              "vm Bus 2 [pu]"],
    ["res_bus.vm_pu.bus3",              "vm Bus 3 [pu]"],
    ["res_bus.va_degree.bus2",          "va Bus 2 [°]"],
    ["res_bus.va_degree.bus3",          "va Bus 3 [°]"],
    ["res_line.i_ka.line",              "I Leitung [kA]"],
    ["res_line.loading_percent.line",   "Auslastung Leitung [%]"],
    ["res_line.pl_mw.line",             "Verluste Leitung [MW]"],
    ["res_trafo.pl_mw.trafo",           "Verluste Trafo [MW]"],
    ["res_trafo.ql_mvar.trafo",         "Blindverluste Trafo [Mvar]"],
    ["res_trafo.loading_percent.trafo", "Auslastung Trafo [%]"],
    ["res_ext_grid.q_mvar.ext",         "Q aus ext. Netz [Mvar]"],
]

def add_derived(df):
    """Ergänzt Spannungsfall und Winkel ohne den 150°-Offset der Schaltgruppe Dyn5
    und rechnet kleine Größen in A, kW und kvar um."""
    df["I Leitung [A]"] = df.pop("I Leitung [kA]") * 1000
    df["Verluste Leitung [kW]"] = df.pop("Verluste Leitung [MW]") * 1000
    df["Verluste Trafo [kW]"] = df.pop("Verluste Trafo [MW]") * 1000
    df["Blindverluste Trafo [kvar]"] = df.pop("Blindverluste Trafo [Mvar]") * 1000
    df["Q aus ext. Netz [kvar]"] = df.pop("Q aus ext. Netz [Mvar]") * 1000
    df["ΔU 2→3 [pu]"] = df["vm Bus 2 [pu]"] - df["vm Bus 3 [pu]"]
    df["Winkel Bus 2 ohne Offset [°]"] = df["va Bus 2 [°]"] + 150
    df["Winkel Bus 3 − Bus 2 [°]"] = df["va Bus 3 [°]"] - df["va Bus 2 [°]"]
    return df.drop(columns=["va Bus 2 [°]", "va Bus 3 [°]"])

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)

## Test 1 – Transformator: `tap_pos`

### Theorie

Der Stufensteller ändert die Übersetzung des Transformators in festen Stufen. Bei diesem Trafo gilt:

| Parameter | Wert | Bedeutung |
|---|---|---|
| `tap_side` | hv | Stufensteller auf der Oberspannungsseite |
| `tap_neutral` | 0 | Stellung mit Nennübersetzung 20/0,4 kV |
| `tap_min` / `tap_max` | −2 / +2 | zulässiger Stellbereich |
| `tap_step_percent` | 2,5 % | Spannungsänderung pro Stufe |
| `tap_changer_type` | Ratio | Längsregler: ändert nur den Betrag, nicht den Winkel |

Jede Stufe ändert die Windungszahl der OS-Wicklung um 2,5 %. Eine **positive** Stufe vergrößert die Übersetzung ü = U_OS / U_US. Bei fester OS-Spannung **sinkt** dadurch die US-Spannung. Eine **negative** Stufe hebt sie an. Der Stufensteller verschiebt das Spannungsniveau des gesamten Niederspannungsnetzes, ohne die Impedanzen nennenswert zu ändern.

### Erwartung

Wenn ich tap_pos erhöhe, erhöht sich die Windungszahl an der Oberspannungsseite, was dazu führt, dass an Bus 2 eine kleinere Spannung anliegt.

Dies wiederum sorgt dafür, dass ein höherer Strom fließt, damit die Last die gleiche Leistung aufnimmt.

Dies führt dazu, dass an der Leitung eine höhere Spannung abfällt, weswegen dort die Auslastung steigt und an Bus 3 die Spannung noch weiter fällt, was wiederum den Strom erhöht.

### Durchführung

In [12]:
df_tap = add_derived(bn.vary_parameter("trafo", "tap_pos", [-2, -1, 1, 2], OUTPUTS))
df_tap

,trafo.tap_pos,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,-2.0000,,True,1.0712,1.0519,48.5552,12.9497,68.9483,0.9156,1.6365,0.3427,5.4598,0.0193,-0.3661,-0.0299
1,-1.0000,,True,1.0437,1.0238,49.8892,12.9447,70.8426,0.9666,1.5622,0.3612,5.4850,0.0198,-0.3858,-0.0315
2,0.0000,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
3,1.0000,,True,0.9925,0.9716,52.5714,12.9401,74.6513,1.0733,1.4311,0.3999,5.5377,0.0209,-0.4267,-0.0349
4,2.0000,,True,0.9687,0.9473,53.9198,13.2616,76.5662,1.1291,1.3731,0.4202,5.5652,0.0215,-0.4481,-0.0367


### Auswertung

Pro Stufe ändert sich die Spannung an Bus 2 um etwa 2,5 %: 1,0437 / 1,0174 ≈ 1,026 ≈ 1/0,975. Das entspricht `tap_step_percent`. Bus 3 verschiebt sich nahezu parallel mit.

Die Erwartung bestätigt sich: Bei `tap_pos = +1` sinkt die Spannung, und weil die Last konstante Leistung bezieht, steigt der Strom (I = P / (√3·U)). Dadurch nehmen Spannungsfall über der Leitung und Auslastung leicht zu. Der Effekt ist aber klein (+2,7 % Strom bei −2,5 % Spannung), weil der Strom umgekehrt proportional zur Spannung ist. Die Rückkopplung schaukelt sich bei dieser geringen Last nicht auf, sondern konvergiert schnell zu einem stabilen Betriebspunkt.

Der Winkel ändert sich kaum, weil `tap_changer_type = Ratio` nur den Betrag der Übersetzung verändert.

## Test 2 – Transformator: `vk_percent` und `vkr_percent`

### Theorie

`vk_percent` ist die **relative Kurzschlussspannung**: die Spannung, die bei kurzgeschlossener Unterspannungsseite nötig ist, damit Nennstrom fließt, bezogen auf die Nennspannung. Sie entspricht dem Betrag der Längsimpedanz des Transformators in pu, bezogen auf `sn_mva` = 0,4 MVA.

`vkr_percent` ist der ohmsche Anteil davon (Kupferverluste). Der induktive Anteil ist v_kx = √(v_k² − v_kr²) und entsteht durch das Streufeld der Wicklungen.

Ein größeres v_k bedeutet mehr lastabhängigen Spannungsfall, begrenzt aber den Kurzschlussstrom stärker. Für Ortsnetztransformatoren dieser Größe sind 4 % und 6 % übliche Werte. Daher wurde der Bereich 4–8 % gewählt. In diesem Versuch bleibt `vkr_percent` = 1,425 % konstant, es ändert sich also nur der induktive Anteil.

### Erwartung

### Durchführung

In [13]:
df_vk = add_derived(bn.vary_parameter("trafo", "vk_percent", [4, 5, 7, 8], OUTPUTS))
df_vk

,trafo.vk_percent,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,4.0000,,True,1.0177,0.9973,51.2132,12.9380,72.7227,1.0186,1.4943,0.2437,5.3743,0.0204,-0.2566,-0.0332
1,5.0000,,True,1.0176,0.9972,51.2206,12.9398,72.7332,1.0189,1.4941,0.3126,5.4433,0.0204,-0.3320,-0.0332
2,6.0000,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
3,7.0000,,True,1.0173,0.9969,51.2352,12.9433,72.7539,1.0195,1.4938,0.4473,5.5780,0.0204,-0.4792,-0.0332
4,8.0000,,True,1.0171,0.9968,51.2425,12.9451,72.7644,1.0197,1.4936,0.5139,5.6447,0.0204,-0.5520,-0.0332


### Auswertung

Eine Änderung von `vk_percent` beeinflusst den Spannungs**betrag** kaum (0,0006 pu über den gesamten Bereich), den **Winkel** an Bus 2 dagegen deutlich: Er verdoppelt sich von 4 % auf 8 % ungefähr (−0,26° → −0,55°).

Erklärung: Da `vkr_percent` konstant bleibt, ändert sich nur der induktive Anteil X. Die Last bezieht nahezu reine Wirkleistung (cos φ ≈ 0,995), deshalb ist der Term X·Q im Längsspannungsfall sehr klein. X wirkt über X·P vor allem auf den Querspannungsfall und damit auf den Winkel.

Die Blindverluste des Trafos (I²·X) steigen proportional zu X. Die Wirkverluste bleiben gleich, weil sich der ohmsche Anteil nicht ändert.

Wegen der geringen Auslastung von 13 % sind alle Effekte am Trafo klein: Der Spannungsfall über einer Impedanz ist proportional zum Strom.

### Ergänzung: `vkr_percent`
Zum Vergleich wird der ohmsche Anteil variiert, `vk_percent` bleibt bei 6 %.

In [14]:
df_vkr = add_derived(bn.vary_parameter("trafo", "vkr_percent", [1.0, 1.25, 1.75, 2.0], OUTPUTS))
df_vkr

,trafo.vkr_percent,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,1.0000,,True,1.0180,0.9976,51.2001,12.9348,72.7042,1.0181,1.4669,0.3856,5.5161,0.0204,-0.4150,-0.0332
1,1.2500,,True,1.0176,0.9973,51.2165,12.9388,72.7274,1.0187,1.4828,0.3827,5.5134,0.0204,-0.4100,-0.0332
2,1.4250,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
3,1.7500,,True,1.0170,0.9966,51.2489,12.9467,72.7735,1.0200,1.5147,0.3748,5.5056,0.0204,-0.3975,-0.0332
4,2.0000,,True,1.0167,0.9963,51.2651,12.9506,72.7964,1.0206,1.5306,0.3696,5.5005,0.0204,-0.3899,-0.0332


Eine Änderung von `vkr_percent` wirkt über R·P direkt auf den Betrag: Bus 2 sinkt von 1,0180 pu (1 %) auf 1,0167 pu (2 %). Gleichzeitig steigen die Kupferverluste. Bei reiner Wirklast beeinflusst also der ohmsche Anteil den Spannungsbetrag stärker als die gesamte Kurzschlussspannung.

## Test 3 – Leitung: `r_ohm_per_km` (Kabellänge 100 m)

### Theorie

`r_ohm_per_km` ist der **Wirkwiderstand** der Leitung pro Kilometer im π-Ersatzschaltbild. Der Gesamtwiderstand ergibt sich aus

R = r · `length_km` / `parallel` = 0,642 Ω/km · 0,1 km / 1 = 0,0642 Ω.

R verursacht Stromwärmeverluste (P_V = 3 · I² · R) und zusammen mit der Wirkleistung den Spannungsfall. Er hängt vom Leitermaterial und vom Querschnitt ab: Ein größerer Querschnitt bedeutet einen kleineren Widerstand. Der Bereich 0,5–0,8 Ω/km entspricht ungefähr den benachbarten Querschnitten eines Aluminiumkabels.

`max_i_ka` ist nur die thermische Stromgrenze und hat keinen Einfluss auf die Impedanz.

### Erwartung

Wenn ich den Widerstand der Leitung erhöhe, fällt dort eine höhere Spannung ab, weswegen an Bus 3 eine geringere Spannung herrscht.

Dies führt dazu, dass ein höherer Strom fließt, damit die Last die gleiche Leistung aufnimmt. Dies erhöht wiederum die an der Leitung abfallende Spannung, was wiederum den Strom erhöht.

### Durchführung

In [15]:
df_r = add_derived(bn.vary_parameter("line", "r_ohm_per_km", [0.5, 0.55, 0.6, 0.7, 0.8], OUTPUTS))
df_r

,line.r_ohm_per_km,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,0.5000,,True,1.0174,1.0016,50.9961,12.8845,72.4145,0.7866,1.4931,0.3769,5.5064,0.0159,-0.4041,-0.0580
1,0.5500,,True,1.0174,1.0000,51.0772,12.9045,72.5297,0.8680,1.4934,0.3781,5.5080,0.0174,-0.4048,-0.0493
2,0.6000,,True,1.0174,0.9984,51.1589,12.9246,72.6456,0.9499,1.4937,0.3793,5.5096,0.0190,-0.4054,-0.0406
3,0.6420,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
4,0.7000,,True,1.0174,0.9952,51.3237,12.9651,72.8797,1.1154,1.4943,0.3817,5.5129,0.0222,-0.4067,-0.0230
5,0.8000,,True,1.0174,0.9920,51.4908,13.0062,73.1169,1.2831,1.4949,0.3841,5.5162,0.0255,-0.4081,-0.0053


### Auswertung

Der Spannungsfall über der Leitung ist nahezu **proportional zu R**: 0,0159 / 0,5 ≈ 0,0204 / 0,642 ≈ 0,0255 / 0,8 ≈ 0,032. Bus 2 bleibt unverändert, weil sich nur der Abschnitt hinter dem Trafo ändert. Die Verluste steigen ebenfalls nahezu linear mit R.

Die erwartete Rückkopplung (niedrigere Spannung → höherer Strom → noch mehr Spannungsfall) ist vorhanden, aber sehr schwach: Der Strom steigt nur von 72,4 A auf 73,1 A (+1 %). Bei einem Spannungsfall von wenigen Prozent ändert sich der Strom ebenfalls nur um wenige Prozent.

Handrechnung zur Plausibilisierung (Baseline):

- ΔU_Strang ≈ I · R = 72,7 A · 0,0642 Ω ≈ 4,67 V → 4,67 V / 231 V ≈ **0,020 pu** ✓
- P_V = 3 · I² · R = 3 · (72,7 A)² · 0,0642 Ω ≈ **1,02 kW** ✓

R ist der wichtigste Leitungsparameter für den Spannungsbetrag, weil das Niederspannungskabel R-dominiert ist und die Last fast reine Wirkleistung bezieht (Term R·P).

## Test 4 – Leitung: `x_ohm_per_km` (Kabellänge 100 m)

### Theorie

`x_ohm_per_km` ist der **induktive Blindwiderstand** pro Kilometer (X = ωL). Er hängt vor allem vom Abstand der Leiter ab. Bei Kabeln liegen die Leiter eng beieinander, deshalb ist X klein (typisch 0,07–0,09 Ω/km). Freileitungen liegen bei etwa 0,3–0,4 Ω/km.

X bestimmt den Blindleistungsbedarf der Leitung (I² · X) und vor allem die Winkeldifferenz zwischen den Leitungsenden. Die Querkapazität `c_nf_per_km` ist bei 100 m Niederspannungskabel vernachlässigbar.

### Erwartung

### Durchführung

In [16]:
df_x = add_derived(bn.vary_parameter("line", "x_ohm_per_km", [0.04, 0.06, 0.1, 0.12], OUTPUTS))
df_x

,line.x_ohm_per_km,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,0.0400,,True,1.0174,0.9972,51.2203,12.9397,72.7328,1.0189,1.4939,0.3802,5.4426,0.0202,-0.4061,0.0427
1,0.0600,,True,1.0174,0.9971,51.2238,12.9406,72.7378,1.0190,1.4939,0.3802,5.4744,0.0203,-0.4060,0.0074
2,0.0830,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
3,0.1000,,True,1.0174,0.9970,51.2309,12.9423,72.7478,1.0193,1.4939,0.3803,5.5380,0.0204,-0.4059,-0.0632
4,0.1200,,True,1.0174,0.9969,51.2345,12.9431,72.7529,1.0194,1.4939,0.3804,5.5698,0.0205,-0.4059,-0.0985


### Auswertung

Der Spannungs**betrag** ändert sich praktisch nicht (0,0003 pu). Die **Winkeldifferenz** über der Leitung ändert sich etwa linear mit X (+0,04° bei 0,04 Ω/km, −0,10° bei 0,12 Ω/km).

Erklärung: Im Längsspannungsfall ist X·Q wegen Q ≈ 0 sehr klein, außerdem ist X beim Kabel gegenüber R gering. Im Querspannungsfall stehen sich X·P und −R·Q gegenüber. In der Baseline heben sich beide Terme fast auf, deshalb liegt die Winkeldifferenz nahe 0°. Bei kleinem X überwiegt −R·Q und der Winkel wird sogar positiv, bei großem X überwiegt X·P.

Im Niederspannungsnetz spielt X für die Spannungshaltung kaum eine Rolle. In X-dominierten Hoch- und Höchstspannungsnetzen ist es umgekehrt: Dort bestimmt X·Q maßgeblich den Spannungsbetrag.

## Test 5 – Last: `p_mw`

### Theorie

`p_mw` ist die **Wirkleistung** der Last in MW. pandapower verwendet das **Verbraucherzählpfeilsystem**: Ein positiver Wert bedeutet Leistungsbezug, ein negativer Wert Einspeisung.

Da alle ZIP-Anteile (`const_z_p_percent`, `const_i_p_percent`) 0 sind, ist die Last eine **Konstantleistungslast**: Sie bezieht unabhängig von der Spannung immer genau P. Sinkt die Spannung, steigt der Strom (I = P / (√3·U)). Dadurch wachsen Spannungsfall und Verluste überproportional. Oberhalb der maximal übertragbaren Leistung gibt es keine Lösung der Leistungsflussgleichungen mehr, und der Lastfluss konvergiert nicht.

### Erwartung

Wenn die Leistung erhöht wird, fließt ein höherer Strom, weswegen an der Leitung eine höhere Spannung abfällt und auch eine höhere Leistung verbraucht wird.

Dies sorgt dafür, dass an Bus 3 die Spannung geringer ist, weswegen ein noch höherer Strom fließt.

Das Ganze sollte sich einpendeln, sofern die Leistungsaufnahme der Last klein genug bleibt.

Außerdem sollte die prozentuale Auslastung steigen.

Wenn ich die aufgenommene Leistung der Last verringere, sollte ein genau gegenteiliger Effekt zu sehen sein.

### Durchführung

In [17]:
df_p = add_derived(bn.vary_parameter("load", "p_mw", [0.02, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6], OUTPUTS))
df_p

,load.p_mw,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,0.0200,,True,1.0185,1.0103,20.7403,5.4354,29.4512,0.1671,1.4183,0.0648,5.0853,0.0082,-0.1578,0.0539
1,0.0500,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
2,0.1000,,True,1.0154,0.9739,104.4958,26.0437,148.3841,4.2406,1.7797,1.5606,7.1078,0.0415,-0.8350,-0.1843
3,0.2000,,True,1.0107,0.9235,220.2039,54.5016,312.6895,18.8314,3.0740,6.8817,14.3153,0.0872,-1.7633,-0.5137
4,0.3000,,True,1.0048,0.8652,352.4913,87.0367,500.5376,48.2536,5.6838,17.5917,28.8291,0.1395,-2.8212,-0.8935
5,0.4000,,True,0.9968,0.7941,512.0773,126.2853,727.1498,101.8366,10.4369,37.0802,55.2452,0.2027,-4.0933,-1.3570
6,0.5000,,True,0.9843,0.6945,731.8429,180.3339,1039.2169,208.0026,19.8544,75.6743,107.5649,0.2899,-5.8378,-2.0062
7,0.6000,,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Auswertung

Die Erwartung bestätigt sich. Der Spannungsfall wächst **stärker als linear** mit P: Vom 0,05- zum 0,2-MW-Fall (Faktor 4) steigt ΔU um den Faktor 4,3, von 0,2 auf 0,5 MW (Faktor 2,5) um den Faktor 3,3. Die Verluste steigen mit I² und damit mehr als quadratisch mit P. Ursache ist die Konstantleistungslast: Bei sinkender Spannung muss mehr Strom fließen, was die Spannung weiter senkt.

Bei kleiner Last pendelt sich das ein, wie erwartet. Mit steigender Last wird die Rückkopplung aber immer stärker:

- Ab etwa 0,1 MW überschreitet der Strom die thermische Grenze des Kabels (142 A).
- Ab etwa 0,25 MW unterschreitet Bus 3 die Grenze von **0,9 pu** (EN 50160).
- Der Fall **0,5 MW konvergiert**, ist aber **physikalisch unzulässig**: Die Leitung wäre 7-fach überlastet, der Trafo mit 180 % über seiner Bemessungsleistung, und den 500 kW Last stehen 208 kW Leitungsverluste gegenüber. Bus 3 liegt bei 0,69 pu.
- Bei **0,6 MW konvergiert der Lastfluss nicht mehr** (Spalte `konvergiert = False`). Die maximal übertragbare Leistung liegt zwischen 0,55 und 0,6 MW (Spitze der P-U-Kurve, „Nasenkurve“). Es existiert kein Betriebspunkt mehr, bei dem die Last ihre Leistung beziehen kann.

Konvergenz bedeutet also nur, dass eine mathematische Lösung existiert, nicht, dass der Betriebspunkt zulässig ist.

## Test 6 – Last: `q_mvar`

### Theorie

`q_mvar` ist die **Blindleistung** der Last in Mvar. Im Verbraucherzählpfeilsystem gilt:

- Q > 0: Die Last bezieht Blindleistung und verhält sich **induktiv** (z. B. Motoren).
- Q < 0: Die Last gibt Blindleistung ab und verhält sich **kapazitiv** (z. B. Kompensationsanlage).

Blindleistung überträgt keine Energie, verursacht aber Strom und damit Verluste. Über den Term X·Q wirkt sie vor allem dort auf den Spannungsbetrag, wo die Netzimpedanz induktiv ist, hier also am Transformator.

### Erwartung

### Durchführung

In [18]:
df_q = add_derived(bn.vary_parameter("load", "q_mvar", [-0.1, -0.05, -0.02, 0.02, 0.05, 0.1], OUTPUTS))
df_q

,load.q_mvar,Baseline,konvergiert,vm Bus 2 [pu],vm Bus 3 [pu],Auslastung Leitung [%],Auslastung Trafo [%],I Leitung [A],Verluste Leitung [kW],Verluste Trafo [kW],Blindverluste Trafo [kvar],Q aus ext. Netz [kvar],ΔU 2→3 [pu],Winkel Bus 2 ohne Offset [°],Winkel Bus 3 − Bus 2 [°]
0,-0.1000,,True,1.0320,1.0166,111.7934,27.6664,158.7466,4.8536,1.8547,1.7735,-97.6001,0.0155,-0.6334,-2.3336
1,-0.0500,,True,1.0252,1.0076,71.3323,17.7962,101.2918,1.9761,1.5896,0.7280,-49.0177,0.0176,-0.5177,-1.2567
2,-0.0200,,True,1.0210,1.0019,54.6321,13.7589,77.5775,1.1591,1.5113,0.4311,-19.4201,0.0190,-0.4547,-0.5948
3,0.0050,x,True,1.0174,0.9970,51.2279,12.9415,72.7436,1.0192,1.4939,0.3803,5.5110,0.0204,-0.4060,-0.0332
4,0.0200,,True,1.0153,0.9940,55.0672,13.8635,78.1954,1.1776,1.5050,0.4379,20.5890,0.0212,-0.3785,0.3083
5,0.0500,,True,1.0109,0.9878,72.7632,18.1435,103.3238,2.0561,1.5771,0.7570,51.0218,0.0231,-0.3276,1.0024
6,0.1000,,True,1.0034,0.9768,116.3433,28.7797,165.2075,5.2567,1.8511,1.9199,102.5985,0.0266,-0.2555,2.1946


### Auswertung

**Bus 2 reagiert stark auf Q:** etwa ±0,015 pu bei ±0,1 Mvar. Der Transformator ist X-dominiert, deshalb wirkt dort der Term X·Q. Zum Vergleich: Eine Erhöhung der Wirkleistung um 0,15 MW (Test 5, 0,05 → 0,2 MW) senkt Bus 2 nur um 0,007 pu.

Bei **kapazitiver** Last (Q < 0) steigt Bus 2 sogar **über die Netzspannung** von 1,02 pu. Die Blindleistung fließt zurück ins vorgelagerte Netz, das externe Netz nimmt bei −0,1 Mvar etwa 98 kvar auf. Das ist der Mechanismus der Spannungsanhebung durch Blindleistungskompensation.

**Über der Leitung** wirkt Q nur schwach auf den Betrag (ΔU 0,016 bzw. 0,027 statt 0,020 pu), weil X beim Kabel klein ist. Dafür kehrt sich die Winkeldifferenz um: Im Querspannungsfall dominiert jetzt der Term −R·Q, und wegen des großen Kabelwiderstands dreht Q den Winkel deutlich (±2,2°).

Blindleistung erhöht in **beiden Richtungen** den Strom. Mit ±0,1 Mvar ist das Kabel in beiden Fällen **überlastet** (112 % bzw. 116 %), obwohl die Wirkleistung nur 0,05 MW beträgt. Eine Spannungsanhebung über Blindleistung kostet also Übertragungskapazität, was im R-dominierten Niederspannungsnetz wenig effizient ist. Werte von ±0,1 Mvar entsprechen einem cos φ von etwa 0,45 und sind für einen realen Verbraucher unrealistisch; realistisch ist etwa der Bereich ±0,02 Mvar (cos φ ≥ 0,93).

## Zusammenfassung der Einzeltests

| Parameter | Wirkt vor allem auf | Grund |
|---|---|---|
| `tap_pos` | Spannungsniveau des ganzen NS-Netzes (±2,5 % pro Stufe) | ändert die Übersetzung |
| `vk_percent` | Winkel an Bus 2 | ändert X des Trafos, Last fast nur P |
| `vkr_percent` | Betrag an Bus 2, Trafoverluste | ändert R des Trafos (R·P) |
| `r_ohm_per_km` | Betrag an Bus 3, Leitungsverluste | Kabel R-dominiert (R·P) |
| `x_ohm_per_km` | Winkel über der Leitung | X·P im Querspannungsfall |
| `p_mw` | Betrag an Bus 3, stark nichtlinear | R·P, Konstantleistungslast |
| `q_mvar` | Betrag an Bus 2, Winkel über der Leitung | X·Q am Trafo, −R·Q am Kabel |

## Aufgabe 6.1 – Vorgelagertes Netz 110 kV

### Variante A: ohne Topologieänderung

#### Netzwerk

Die Aufgabe fordert, das Netz zunächst **ohne Topologieänderung** auf ein vorgelagertes 110-kV-Netz umzustellen. Das Netz behält also seine drei Busse und einen Transformator (`build_110kV_net_simple()`). Geändert werden alle Parameter, die mit der Nennspannung des vorgelagerten Netzes verbunden sind:

| Element | Parameter | Alt | Neu | Begründung |
|---|---|---|---|---|
| Bus 1 | `vn_kv` | 20 kV | 110 kV | Nennspannung des vorgelagerten Netzes |
| Trafo | `vn_hv_kv` | 20 kV | 110 kV | OS-Bemessungsspannung muss zur Busspannung passen, sonst entsteht eine Fehlanpassung der Übersetzung |
| Trafo | `std_type`, `name` | 0.4 MVA 20/0.4 kV | – / „Trafo 110/0.4 kV“ | Der Standardtyp stimmt nicht mehr |
| ext_grid | `vm_pu` | 1,02 pu | 1,02 pu | in pu unverändert, entspricht jetzt 112,2 kV |

#### Datenstruktur

In [19]:
net, idx = bn.build_110kV_net_simple()
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,110.0000,b,None,True,None
1,Bus 2,0.4000,b,None,True,None
2,Bus 3,0.4000,b,None,True,None


In [20]:
net.trafo[["name", "std_type", "sn_mva", "vn_hv_kv", "vn_lv_kv", "vk_percent", "vkr_percent"]]

,name,std_type,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent
0,Trafo 110/0.4 kV,None,0.4000,110.0000,0.4000,6.0000,1.4250


#### Ergebnisse

In [21]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.0200,0.0000,-0.0525,-0.0055
1,1.0174,-150.4060,0.0000,0.0000
2,0.9970,-150.4392,0.0500,0.0050


In [22]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.0525,0.0055,-0.0510,-0.0051,0.0015,0.0004,0.0003,0.0727,1.0200,0.0000,1.0174,-150.4060,12.9415


#### Auswertung

Alle Spannungen an Bus 2 und 3 in pu sind **identisch zur Baseline** (1,0174 / 0,9970 pu). Nur der OS-Strom des Trafos sinkt von 1,49 A auf 0,27 A, also um den Faktor 110/20 = 5,5.

Erklärung: Die Trafo-Kennwerte `vk_percent`, `vkr_percent`, `pfe_kw` und `i0_percent` sind relative Größen, bezogen auf Bemessungsspannung und -leistung. In pu ändert sich deshalb nichts. Das zeigt den Vorteil der pu-Darstellung.

Einschränkung: Ein direkter 110/0,4-kV-Transformator mit 0,4 MVA ist technisch unüblich. Reale Netze haben zwischen 110 kV und 0,4 kV immer eine Mittelspannungsebene. Variante A ist deshalb eine Modellbetrachtung, Variante B die realistische Erweiterung.

### Variante B: mit zusätzlicher Spannungsebene

Externes Netz → Bus 1 (110 kV) → Transformator 1 → Bus 2 (20 kV) → Transformator 2 → Bus 3 (0,4 kV) → Kabel → Bus 4 (0,4 kV) → Last

Zwischen dem externen Netz und dem Bus des Mittelspannungsnetzes wurden ein neuer Bus `Bus 1` (110 kV) und ein Transformator `25 MVA 110/20 kV` hinzugefügt.

#### Netz

| Komponente | pandapower-Element | Wichtige Daten | Funktion |
|---|---|---|---|
| Externes Netz | `ext_grid` an Bus 1 | `vm_pu` = 1,02, Winkel 0° | Vorgelagertes Hochspannungsnetz als starre Spannungsquelle. Es ist der Slack-Bus: Er gibt Spannung und Winkelreferenz vor und gleicht die Leistungsbilanz (Last + Verluste) aus. |
| Bus 1 | `bus`, 110 kV | Nennspannung 110 kV | Hochspannungs-Sammelschiene, Oberspannungsseite von Trafo 1 |
| Transformator 1 | `trafo`, Typ „25 MVA 110/20 kV“ | S_r = 25 MVA, v_k = 12 %, v_kr = 0,41 %, Schaltgruppe YNd5, Stufensteller ±9 × 1,5 % auf der OS-Seite | HS/MS-Umspanntransformator, setzt 110 kV auf 20 kV um |
| Bus 2 | `bus`, 20 kV | Nennspannung 20 kV | Mittelspannungs-Sammelschiene, Oberspannungsseite von Trafo 2 |
| Transformator 2 | `trafo`, Typ „0.4 MVA 20/0.4 kV“ | S_r = 0,4 MVA, v_k = 6 %, v_kr = 1,425 %, Schaltgruppe Dyn5, Stufensteller ±2 × 2,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 3 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Niederspannungs-Sammelschiene der Ortsnetzstation |
| Kabel | `line`, Typ „NAYY 4x50 SE“ | 0,1 km, r = 0,642 Ω/km, x = 0,083 Ω/km, c = 210 nF/km, I_max = 142 A, 1 System | Niederspannungs-Erdkabel (Aluminium, 50 mm²) zum Verbraucher |
| Bus 4 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Netzknoten am Ende des Kabels, Anschlusspunkt des Verbrauchers |
| Last | `load` an Bus 4 | P = 0,05 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |

#### Datenstruktur

In [23]:
net, idx = bn.build_110kV_net()

In [24]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,110.0000,b,None,True,None
1,Bus 2,20.0000,b,None,True,None
2,Bus 3,0.4000,b,None,True,None
3,Bus 4,0.4000,b,None,True,None


In [25]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,i0_percent,shift_degree,tap_side,tap_neutral,tap_min,tap_max,tap_step_percent,tap_step_degree,tap_pos,tap_changer_type,id_characteristic_table,tap_dependency_table,parallel,df,in_service,vector_group,oltc
0,Trafo 1,25 MVA 110/20 kV,0,1,25.0000,110.0000,20.0000,12.0000,0.4100,14.0000,0.0700,150.0000,hv,0.0000,-9.0000,9.0000,1.5000,0.0000,0.0000,Ratio,<NA>,False,1,1.0000,True,YNd5,False
1,Trafo 2,0.4 MVA 20/0.4 kV,1,2,0.4000,20.0000,0.4000,6.0000,1.4250,1.3500,0.3375,150.0000,hv,0.0000,-2.0000,2.0000,2.5000,0.0000,0.0000,Ratio,<NA>,False,1,1.0000,True,Dyn5,False


In [26]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line,NAYY 4x50 SE,2,3,0.1000,0.6420,0.0830,210.0000,0.0000,0.1420,1.0000,1,cs,True,None


In [27]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type
0,Load,3,0.0500,0.0050,0.0000,0.0000,0.0000,0.0000,NaN,1.0000,True,wye


#### Ergebnisse

In [28]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.0200,0.0000,-0.0671,-0.0165
1,1.0199,-150.0157,0.0000,0.0000
2,1.0174,59.5783,0.0000,0.0000
3,0.9970,59.5451,0.0500,0.0050


In [29]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.0671,0.0165,-0.0525,-0.0055,0.0146,0.0109,0.0004,0.0015,1.0200,0.0000,1.0199,-150.0157,0.2708
1,0.0525,0.0055,-0.0510,-0.0051,0.0015,0.0004,0.0015,0.0727,1.0199,-150.0157,1.0174,59.5783,12.9423


In [30]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.0510,0.0051,-0.0500,-0.0050,0.0010,0.0001,0.0727,0.0727,0.0727,1.0174,59.5783,0.9970,59.5451,51.2311


In [31]:
net.res_load

,p_mw,q_mvar
0,0.0500,0.0050


#### Auswertung

- Die NS-Spannungen sind praktisch identisch mit der Baseline. Der 25-MVA-Trafo ist mit 0,27 % kaum belastet. Trotz v_k = 12 % fällt über ihm nur 0,00006 pu ab. Das vorgelagerte Netz bleibt aus Sicht der Last nahezu „starr“.
- Die Verluste steigen deutlich: Trafo 1 hat 14,6 kW Verluste, fast ausschließlich Eisenverluste (`pfe_kw` = 14 kW). Bei dieser kleinen Last ist das mehr als ein Viertel der Lastleistung. Der Bezug aus dem Netz steigt von 52,5 kW auf 67,1 kW.
- Die Winkel von etwa +59,6° an Bus 3 und 4 sind keine Spannungsfälle. Beide Trafos haben eine Phasendrehung von 150° (YNd5 und Dyn5), zusammen 300°. pandapower stellt das als −300° ≡ +60° dar. Der lastbedingte Anteil ist nur 59,58° − 60° = −0,42°.

## Aufgabe 6.2 – Zweiter Lastabgang an Bus 3

### Netzwerk

Externes Netz → Bus 1 (20 kV) → Transformator → Bus 2 (0,4 kV) → Kabel → Bus 3 (0,4 kV) → Last 1 + Last 2

An Bus 3 wurde eine weitere Last `Load 2` angeschlossen.

| Komponente | pandapower-Element | Wichtige Daten | Funktion |
|---|---|---|---|
| Externes Netz | `ext_grid` an Bus 1 | `vm_pu` = 1,02, Winkel 0° | Vorgelagertes Mittelspannungsnetz als starre Spannungsquelle. Es ist der Slack-Bus: Er gibt Spannung und Winkelreferenz vor und gleicht die Leistungsbilanz (Last + Verluste) aus. |
| Bus 1 | `bus`, 20 kV | Nennspannung 20 kV | Mittelspannungs-Sammelschiene, Oberspannungsseite des Trafos |
| Transformator | `trafo`, Typ „0.4 MVA 20/0.4 kV“ | S_r = 0,4 MVA, v_k = 6 %, v_kr = 1,425 %, Schaltgruppe Dyn5, Stufensteller ±2 × 2,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 2 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Niederspannungs-Sammelschiene der Ortsnetzstation |
| Kabel | `line`, Typ „NAYY 4x50 SE“ | 0,1 km, r = 0,642 Ω/km, x = 0,083 Ω/km, c = 210 nF/km, I_max = 142 A, 1 System | Niederspannungs-Erdkabel (Aluminium, 50 mm²) zum Verbraucher |
| Bus 3 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Netzknoten am Ende des Kabels, Anschlusspunkt des Verbrauchers |
| Last 1 | `load` an Bus 3 | P = 0,05 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |
| Last 2 | `load` an Bus 3 | P = 0,02 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |

### Datenstruktur

In [32]:
net, idx = bn.build_2_load_net()

In [33]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,20.0000,b,None,True,None
1,Bus 2,0.4000,b,None,True,None
2,Bus 3,0.4000,b,None,True,None


In [34]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,i0_percent,shift_degree,tap_side,tap_neutral,tap_min,tap_max,tap_step_percent,tap_step_degree,tap_pos,tap_changer_type,id_characteristic_table,tap_dependency_table,parallel,df,in_service,vector_group,oltc
0,Trafo,0.4 MVA 20/0.4 kV,0,1,0.4000,20.0000,0.4000,6.0000,1.4250,1.3500,0.3375,150.0000,hv,0.0000,-2.0000,2.0000,2.5000,0.0000,0.0000,Ratio,<NA>,False,1,1.0000,True,Dyn5,False


In [35]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line,NAYY 4x50 SE,1,2,0.1000,0.6420,0.0830,210.0000,0.0000,0.1420,1.0000,1,cs,True,None


In [36]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type
0,Load 1,2,0.0500,0.0050,0.0000,0.0000,0.0000,0.0000,NaN,1.0000,True,wye
1,Load 2,2,0.0200,0.0050,0.0000,0.0000,0.0000,0.0000,NaN,1.0000,True,wye


### Ergebnisse

In [37]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.0200,0.0000,-0.0736,-0.0110
1,1.0159,-150.5660,0.0000,0.0000
2,0.9869,-150.5442,0.0700,0.0100


In [38]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.0736,0.0110,-0.0721,-0.0103,0.0016,0.0008,0.0021,0.1034,1.0200,0.0000,1.0159,-150.5660,18.2515


In [39]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.0721,0.0103,-0.0700,-0.0100,0.0021,0.0003,0.1034,0.1034,0.1034,1.0159,-150.5660,0.9869,-150.5442,72.8256


In [40]:
net.res_load

,p_mw,q_mvar
0,0.0500,0.0050
1,0.0200,0.0050


### Vergleich mit der Baseline

In [41]:
def kennwerte(net):
    return {
        "P Last gesamt [MW]": net.res_load.p_mw.sum(),
        "vm Bus 2 [pu]": net.res_bus.vm_pu[1],
        "vm Bus 3 [pu]": net.res_bus.vm_pu[2],
        "ΔU 2→3 [pu]": net.res_bus.vm_pu[1] - net.res_bus.vm_pu[2],
        "I Leitung [A]": net.res_line.i_ka[0] * 1000,
        "Auslastung Leitung [%]": net.res_line.loading_percent[0],
        "Verluste Leitung [kW]": net.res_line.pl_mw[0] * 1000,
        "Auslastung Trafo [%]": net.res_trafo.loading_percent[0],
    }

net_base, _ = bn.build_base_net()
pp.runpp(net_base)
net_2l, _ = bn.build_2_load_net()
pp.runpp(net_2l)
pd.DataFrame({"Baseline (1 Last)": kennwerte(net_base), "2 Lasten": kennwerte(net_2l)})

,Baseline (1 Last),2 Lasten
P Last gesamt [MW],0.0500,0.0700
vm Bus 2 [pu],1.0174,1.0159
vm Bus 3 [pu],0.9970,0.9869
ΔU 2→3 [pu],0.0204,0.0290
I Leitung [A],72.7436,103.4124
Auslastung Leitung [%],51.2279,72.8256
Verluste Leitung [kW],1.0192,2.0597
Auslastung Trafo [%],12.9415,18.2515


### Auswertung

- Zwei Lasten am selben Bus wirken im Lastfluss wie **eine Last mit der Summe** der Leistungen (0,07 MW / 0,01 Mvar). pandapower addiert die Leistungen aller Elemente an einem Bus.
- Strom und Spannungsfall über der Leitung steigen etwa proportional zur Gesamtleistung (+42 %): Bus 3 sinkt von 0,9970 auf 0,9869 pu.
- Die Leitungsverluste verdoppeln sich, weil sie mit I² wachsen: 1,42² ≈ 2,0.
- Alle Grenzwerte werden eingehalten. Die Leitung ist zu 73 % ausgelastet und hat nur noch etwa 27 % Reserve.